# 第8回 Transformer — 文のどこに注目するかを、AI が決める

**このノートブックは第8回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 離れた主語を見て `is` / `are` を決められるか確かめる
2. 144個の注意ヘッドの中身をのぞく(平均と個別で、まったく違う図になります)
3. 語順を壊して、位置情報が効いていることを確かめる
4. 教科書の有名な図を、自分で再現してみる(私はできませんでした)

所要時間:実習 約25分 / 課題 約40分

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。
最初の2つのセルは、合わせて1〜2分かかります。

In [ ]:
# 準備(このセルを最初に1回だけ実行。1〜2分かかります)
!pip install -q transformers japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import torch, numpy as np
import matplotlib.pyplot as plt, japanize_matplotlib
from transformers import AutoTokenizer, AutoModelForCausalLM
plt.rcParams["font.size"] = 12
print("準備できました")

In [ ]:
# GPT-2 を読み込む(注意の重みを取り出せる設定にします)
トークナイザ = AutoTokenizer.from_pretrained("gpt2")
モデル = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager")
モデル.eval()
print(f"層の数     : {モデル.config.n_layer}")
print(f"ヘッドの数 : {モデル.config.n_head}(1層あたり)")
print(f"合計       : {モデル.config.n_layer * モデル.config.n_head} 組の注意")

## 1. 遠くの語を見ないと解けない問題

英語の文法では、動詞の形は **主語** に合わせます。
`The key to the cabinets ___` の主語は `key`(単数)なので、`is` が入ります。

でも動詞の直前にあるのは `cabinets`(複数)です。
AI は、**隣の語ではなく離れた主語**を見て決められるでしょうか。

In [ ]:
# ===== ① 遠くの語を見ないと解けない問題 =====
def 次の語(文, 上位=5):
    ids = トークナイザ.encode(文, return_tensors="pt")
    with torch.no_grad():
        確率 = torch.softmax(モデル(ids).logits[0, -1], dim=-1)
    上 = torch.topk(確率, 上位)
    語 = [トークナイザ.decode([i]).strip() or "␣" for i in 上.indices]
    値 = [float(v) * 100 for v in 上.values]
    print(f"「{文} ___」")
    for w, v in zip(語, 値):
        印 = "  ←" if w in ("is", "are", "was", "were") else ""
        print(f"    {w:<12}{v:5.1f}%{印}")
    print()
    return dict(zip(語, 値))

a = 次の語("The key to the cabinets")     # 主語は key(単数)
b = 次の語("The keys to the cabinet")     # 主語は keys(複数)

print("動詞の直前にあるのは cabinets / cabinet です。")
print("でも動詞の形を決めているのは、3語前の key / keys のほうです。")
print("隣ではなく、離れた語を見ている ── これが「注意」のはたらきです。")

In [ ]:
# 主語と動詞の距離を伸ばすと、どこまで保たれるか
# あいだに「わざと逆の数の名詞」を挟んで、引っかかるか試します。
検査 = {
    "主語 key(単数)/ 挟むのは複数形": [
        "The key to the cabinets",
        "The key to the cabinets and the drawers",
        "The key to the cabinets and the drawers in the offices",
        "The key to the cabinets and the drawers in the offices of the buildings",
    ],
    "主語 keys(複数)/ 挟むのは単数形": [
        "The keys to the cabinet",
        "The keys to the cabinet and the drawer",
        "The keys to the cabinet and the drawer in the office",
        "The keys to the cabinet and the drawer in the office of the building",
    ],
}
IS  = トークナイザ.encode(" is")[0]
ARE = トークナイザ.encode(" are")[0]

for 見出し, 文たち in 検査.items():
    正解は = "is" if "単数" in 見出し.split("/")[0] else "are"
    print(f"【{見出し}】 正解は {正解は}")
    print(f"{'挟んだ語数':>10}{'is':>9}{'are':>9}   順位")
    for 文 in 文たち:
        ids = トークナイザ.encode(文, return_tensors="pt")
        with torch.no_grad():
            p = torch.softmax(モデル(ids).logits[0, -1], dim=-1)
        i_, a_ = float(p[IS]) * 100, float(p[ARE]) * 100
        当たり = (i_ > a_) if 正解は == "is" else (a_ > i_)
        print(f"{len(文.split())-2:>10}{i_:>8.2f}%{a_:>8.2f}%   {'○ 正しい' if 当たり else '× 逆転'}")
    print()

print("見るところは2つあります ── 順位が逆転するか、そして確率(自信)がどう変わるか。")
print("この2つは別物です。順位が正しくても、自信が薄れていることがあります。")
print("※ 人間も『The key to the cabinets are』と言い間違えます")
print("   (『引き寄せ効果』として言語学で研究されてきた現象です)")

## 2. 144個のヘッドを、のぞいてみる

**注意**とは「どの語を何割の重みで見るか」を決める仕組みです。重みの合計は必ず 1 になります。
GPT-2 には 12層 × 12ヘッド = **144組**の注意があります。実際の数字を見てみましょう。

In [ ]:
# ===== ② 144個のヘッドを、のぞいてみる =====
def 注意を取る(文):
    ids = トークナイザ.encode(文, return_tensors="pt")
    with torch.no_grad():
        出力 = モデル(ids, output_attentions=True)
    語 = [トークナイザ.decode([i]).strip() for i in ids[0]]
    # (層, ヘッド, 見る位置, 見られる位置)
    重み = torch.stack(出力.attentions)[:, 0]
    return 語, 重み

語, 重み = 注意を取る("The key to the cabinets is")
i = 語.index("is")          # 「is」の位置から見る
print("トークン:", 語)
print(f"\n「is」から各語への重み(144ヘッド全部の平均):")
平均 = 重み.mean(dim=(0, 1))[i]
for j, w in enumerate(語[:i+1]):
    print(f"   {w:<12}{float(平均[j]):.4f}  {'■' * int(float(平均[j]) * 40)}")
print(f"\nいちばん大きいのは… 文頭の「{語[int(平均[:i+1].argmax())]}」です。")
print("肝心の key はほとんど見ていないように見えます。おかしいですね。")

> 平均を見ると、肝心の `key` はほとんど見ていないように見えます。
> **でも、それは平均だからです。** ヘッドを1つずつ見ると、話が変わります。

In [ ]:
# ヘッドを1つずつ見ると、話が変わる
key = 語.index("key")
候補 = []
for L in range(重み.shape[0]):
    for H in range(重み.shape[1]):
        候補.append((float(重み[L, H, i, key]), L + 1, H + 1))
候補.sort(reverse=True)

print("「key」をいちばん強く見ているヘッド トップ5:")
for v, L, H in 候補[:5]:
    行 = 重み[L-1, H-1, i, :i+1]
    最大語 = 語[int(行.argmax())]
    print(f"   第{L:>2}層 ヘッド{H:>2}: key への重み {v:.3f}"
          f"   (このヘッドの最大は「{最大語}」{float(行.max()):.3f})")

print()
print(f"144個の平均では key は {float(平均[key]):.4f} でしたが、")
print(f"第{候補[0][1]}層ヘッド{候補[0][2]} だけを見ると {候補[0][0]:.3f} です。")
print("1個の特徴的なヘッドが、143個の凡庸なヘッドに薄められていたのです。")

In [ ]:
# 2枚の図を並べる ── 同じモデル、同じ入力、違う見せかた
最良 = 候補[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
語表示 = 語[:i+1]

for ax, (値, 題) in zip(axes, [
    (重み.mean(dim=(0, 1))[i, :i+1].numpy(), "144ヘッドの平均"),
    (重み[最良[1]-1, 最良[2]-1, i, :i+1].numpy(), f"第{最良[1]}層 ヘッド{最良[2]} だけ"),
]):
    色 = ["#E08A2B" if w == "key" else "#3A46A8" for w in 語表示]
    ax.bar(range(len(語表示)), 値, color=色)
    ax.set_xticks(range(len(語表示))); ax.set_xticklabels(語表示, fontsize=10)
    ax.set_ylabel("重み"); ax.set_title(題); ax.set_ylim(0, 0.75)
    ax.grid(alpha=.3, axis="y")
plt.suptitle("同じモデルの同じ入力から、正反対の印象の図が作れます", fontsize=12)
plt.tight_layout(); plt.show()

print("左を見た人は「文頭ばかり見ている」と言い、右を見た人は「主語を見ている」と言います。")
print("どちらも嘘ではありません。だから図を見たら『どのヘッドか』を確かめてください。")

## 3. 語順は、本当に効いているのか

注意は「全部の語を一度に見る」仕組みでした。そのままでは語の並び順が分かりません。
そこで各語の数字に「何番目か」という情報を足しておきます(**位置情報**)。

本当に効いているのか、語順を壊して確かめます。

In [ ]:
# ===== ③ 語順は、本当に効いているのか =====
def 尤度(文):
    """loss が小さいほど「ありそうな文」"""
    ids = トークナイザ.encode(文, return_tensors="pt")
    with torch.no_grad():
        return float(モデル(ids, labels=ids).loss)

import random
もとの文 = "The dog chased the cat and the cat ran away"
random.seed(0)
語リスト = もとの文.split()

print(f"{'loss':>7}   文")
print("-" * 60)
print(f"{尤度(もとの文):>7.3f}   {もとの文}   ← 正しい語順")
for k in range(3):
    s = 語リスト[:]; random.shuffle(s); s = " ".join(s)
    print(f"{尤度(s):>7.3f}   {s}")

print()
print("語の集合はまったく同じです。順番だけが違います。それでも loss はほぼ2倍。")
print("→ 位置情報(何番目の語か)は、確かに効いています。")

In [ ]:
# 語は同じで、順番だけ違う文を比べる
組 = [("I have to go", "To go I have"),
      ("The dog bit the man", "The man bit the dog"),
      ("The cat sat on the mat", "The mat sat on the cat")]
for a, b in 組:
    la, lb = 尤度(a), 尤度(b)
    print(f"  {la:.3f}  「{a}」")
    print(f"  {lb:.3f}  「{b}」    差 {abs(la-lb):.3f}")
    print()
print("差が大きい組ほど、「片方だけが自然な文」ということです。")
print("どちらも英語として成立する組は、差が小さくなります。")

## 4. 有名な図を、自分で再現してみる

ここからが今日いちばん面白いところです。

Transformer の解説には、ほぼ必ずこの例が出てきます ——
「`it` が指すものが `tired` / `wide` で切り替わり、注意の重みもそう動く」。

**本当でしょうか。自分で確かめましょう。**

In [ ]:
# ===== ④ 有名な図を、自分で再現してみる =====
# Transformer の解説には必ずこの例が出てきます。
#   「The animal didn't cross the street because it was too tired.」
#   最後を tired → wide に変えると、「it」が指すものが animal → street に変わる。
# 本当にそうなるでしょうか。

def itが見ているもの(文):
    語, 重み = 注意を取る(文)
    i = 語.index("it")
    平均 = 重み.mean(dim=(0, 1))[i]
    a = float(平均[語.index("animal")])
    s = float(平均[語.index("street")])
    return a, s

for 語尾 in ["tired", "wide"]:
    文 = f"The animal didn't cross the street because it was too {語尾}"
    a, s = itが見ているもの(文)
    print(f"…too {語尾:<6}→  animal {a:.4f}   street {s:.4f}   "
          f"強いのは {'animal' if a > s else 'street'}")

print()
print("数字が完全に同じになったはずです。理由を考えてみてください。")
print("(ヒント:GPT-2 は左から右へしか読めません。「it」を処理する時点で、")
print(" 文の最後の tired / wide は、まだ見えていません)")

In [ ]:
# 前も後ろも見られる BERT なら再現できるか(初回は1分ほどダウンロード)
from transformers import AutoModel
bt = AutoTokenizer.from_pretrained("bert-base-uncased")
bm = AutoModel.from_pretrained("bert-base-uncased", attn_implementation="eager"); bm.eval()

for 語尾 in ["tired", "wide"]:
    文 = f"the animal did not cross the street because it was too {語尾}"
    enc = bt(文, return_tensors="pt")
    with torch.no_grad():
        出力 = bm(**enc, output_attentions=True)
    語B = bt.convert_ids_to_tokens(enc["input_ids"][0])
    A = torch.stack(出力.attentions)[:, 0]
    i = 語B.index("it")
    平均 = A.mean(dim=(0, 1))[i]
    a = float(平均[語B.index("animal")]); s = float(平均[語B.index("street")])
    print(f"…too {語尾:<6}→  animal {a:.4f}   street {s:.4f}   "
          f"強いのは {'animal' if a > s else 'street'}")

print()
print("BERT は前も後ろも見られます。それでも切り替わりましたか?")
print("元の図は2017年の翻訳モデルの特定の層・特定のヘッドを選んで描かれたもので、")
print("Transformer 一般の性質ではありません。引用されるうちに条件が落ちた例です。")

> **今日いちばん持ち帰ってほしいこと**
>
> 同じモデルの同じ入力から、正反対の印象の図が2枚作れました。
> 注意の図を見たら、必ず確かめてください ── **どのモデルか、どの層か、どのヘッドか、平均か個別か。**
> それが書かれていない図は、証拠として弱いと考えてください。

## 5. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# (a) 自分で作った文で is / are を予測させる
私の文 = [
    "The book on the shelves",
    "The books on the shelf",
    "The teacher of the students",
]
for 文 in 私の文:
    次の語(文, 上位=4)

# (b) 自分の文で注意を見る(語を書きかえてください)
私の文2 = "The report from the managers is"
語2, 重み2 = 注意を取る(私の文2)
i2 = len(語2) - 1
print("トークン:", 語2)
平均2 = 重み2.mean(dim=(0, 1))[i2]
for j, w in enumerate(語2[:i2+1]):
    print(f"   {w:<12}{float(平均2[j]):.4f}")

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:注意の重みを読む
次の重みを読んで答えなさい(計算は電卓可)。

```
The 0.08 / reports 0.35 / from 0.04 / the 0.03 / committee 0.20 / ___ 0.30
```

- (a) 合計が 1 になることを確かめなさい
- (b) 空欄は `is` か `are` か。理由も書きなさい
- (c) 「reports が答えを決めた」と言い切れますか。
  言い切れないとしたら、**確かめるには何をすればよいですか**

### 課題2:ノートブックを動かして報告する
- (a) 主語と動詞の距離を伸ばして、正解がどこで崩れるかの表
- (b) 「144ヘッドの平均」と「第4層ヘッド12」の2枚の図を貼り、違いを説明する

### 課題3:次の主張を批判する(300字以上)

> 「この AI がなぜその判断をしたかは、注意の重みを見れば分かります。
> 重みがいちばん大きい語が、判断の根拠です。これで説明責任は果たせます。」

考える材料:
- 今日の実測で、144ヘッドの平均でいちばん重かったのはどの語でしたか
- 「注意の重み」と言うとき、144通りのうちのどれを指していますか
- 重みが高いことと、出力を決めたことは同じですか(相関と因果)
- そもそも、内部の数字を見せれば説明責任を果たしたことになりますか

**加点**:「では何をもって説明とすべきか」まで踏み込めると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。
課題2で予想と違う結果が出たら、それも書いてください。むしろ面白い発見です。